# 02. 주식 트레이딩 PPO (Sim-to-Finance)
> **Google Colab T4** 에서 위에서 아래로 순서대로 셀을 실행하세요.

## 1. 패키지 설치 및 레포 클론

In [ ]:
!pip install -q stable-baselines3==2.3.2 gymnasium==0.29.1 shimmy==1.3.0 tensorboard>=2.9.1 yfinance==0.2.37

In [ ]:
import os, sys

REPO = 'rl-sim-to-finance'
if not os.path.exists(f'/content/{REPO}'):
    # ⬇ 본인 GitHub URL로 교체하세요
    !git clone https://github.com/YOUR_USERNAME/rl-sim-to-finance.git /content/{REPO}

ROOT = f'/content/{REPO}'
for d in [ROOT, f'{ROOT}/data', f'{ROOT}/src/env', f'{ROOT}/agents', f'{ROOT}/utils']:
    if d not in sys.path:
        sys.path.insert(0, d)

os.makedirs('/content/results/models', exist_ok=True)
os.makedirs('/content/results/plots', exist_ok=True)
print('환경 준비 완료')

## 2. 주가 데이터 수집

In [ ]:
from fetcher import fetch_stock_data, add_features, train_test_split_df
import numpy as np

TICKER  = 'AAPL'   # 원하는 티커로 변경 가능 (예: '005930.KS')
START   = '2018-01-01'
END     = '2024-01-01'

df_raw  = fetch_stock_data(TICKER, START, END)
df      = add_features(df_raw, window=20)
df_train, df_test = train_test_split_df(df, test_ratio=0.2)

print(f'학습 데이터: {len(df_train)}일  |  테스트 데이터: {len(df_test)}일')
df.tail(3)

## 3. 트레이딩 환경 확인

In [ ]:
from trading_env import TradingEnv

prices_train = df_train['Close'].values.astype('float32')
env = TradingEnv(prices_train, window_size=20, initial_balance=10_000)

obs, _ = env.reset()
print('Observation shape:', obs.shape)
print('Action space:', env.action_space)

# 랜덤 에이전트 1 에피소드
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
print(f'랜덤 에이전트 최종 포트폴리오: ${info["portfolio_value"]:,.2f}')

## 4. PPO 학습

In [ ]:
from ppo_agent import PPOTradingAgent
from trading_env import TradingEnv
import numpy as np

prices_train = df_train['Close'].values.astype('float32')

agent = PPOTradingAgent(
    env_fn=lambda: TradingEnv(prices_train, window_size=20, initial_balance=10_000),
    learning_rate=3e-4,
    n_steps=512,
    batch_size=64,
    n_epochs=10,
    verbose=1,
    device='auto',
)

agent.train(total_timesteps=200_000)
agent.save('/content/results/models/ppo_trading')
print('학습 완료')

## 5. 학습 곡선 시각화

In [ ]:
import sys
sys.path.insert(0, f'{ROOT}/utils')
from visualizer import plot_learning_curve

fig = plot_learning_curve(
    agent.reward_callback.episode_rewards,
    title=f'{TICKER} PPO Learning Curve',
    window=30,
    save_path='/content/results/plots/trading_curve.png',
)
fig.show()

## 6. 테스트 데이터로 백테스트

In [ ]:
from trading_env import TradingEnv
from visualizer import plot_portfolio, plot_action_distribution
import numpy as np

prices_test = df_test['Close'].values.astype('float32')
test_env = TradingEnv(prices_test, window_size=20, initial_balance=10_000)

obs, _ = test_env.reset()
done = False
portfolio_values = [test_env.initial_balance]

while not done:
    action = agent.predict(obs)
    obs, reward, terminated, truncated, info = test_env.step(action)
    portfolio_values.append(info['portfolio_value'])
    done = terminated or truncated

final_value = portfolio_values[-1]
ret = (final_value - 10_000) / 10_000 * 100
print(f'테스트 수익률: {ret:+.2f}%  |  최종 포트폴리오: ${final_value:,.2f}')

fig1 = plot_portfolio(
    prices_test, np.array(portfolio_values), test_env.trade_log,
    ticker=TICKER, initial_balance=10_000,
    save_path='/content/results/plots/trading_result.png',
)
fig1.show()

fig2 = plot_action_distribution(test_env.trade_log, save_path='/content/results/plots/action_dist.png')
fig2.show()

---
다음 → **app/streamlit_app.py** 를 로컬 또는 Colab 터널링으로 실행해 인터랙티브 데모를 확인하세요.